# Fingerprint Agreement on the Validation Library

## Setup

In [ ]:
from setup import *

from rdkit import DataStructs
from torch.utils.data import random_split

## Compute Validation Correlations

In [ ]:
seed = 0
n_queries = 1000
n_bootstrap = 2000
query_batch_size = 16

dataset = ConformerEdgeDataset(str(CHEMBL_DIR), num_conformers=16, sample_per_batch=1, use_molecule_properties=False, randomize=False, preload=False)
_, validation = random_split(dataset, [0.95, 0.05], generator=torch.Generator().manual_seed(seed))
library = pd.DataFrame({"ID": torch.cat([data["id"].reshape(-1).cpu() for data in dataset.data]).numpy()[validation.indices]}).sort_values("ID").reset_index(drop=True)
del dataset, _, validation

caches = {name: load_or_compute_embeddings(EMBEDDING_DIR / f"embeddings_{name}.pt", CHECKPOINTS[name], CHEMBL_DIR, DEVICE, mmap=True) for name in ("ANT", "NT", "nopre")}
_, val_split = random_split(range(len(caches["ANT"]["id"])), [0.95, 0.05], generator=torch.Generator().manual_seed(seed))
sample_indices = np.sort(np.random.default_rng(seed).choice(val_split.indices, size=min(n_queries, len(val_split)), replace=False))
query_ids = caches["ANT"]["id"][sample_indices]
del val_split, _, sample_indices
for name, cache in caches.items():
    cache_ids = pd.Index(cache["id"].reshape(-1).numpy())
    if not cache_ids.is_unique or len(cache_ids) != len(cache["embeddings"]): raise ValueError(f"{name}: expected one embedding per unique molecule ID")
    library[name] = cache_ids.get_indexer(library["ID"])
library = library[(library[list(caches)] >= 0).all(axis=1)].copy()
if library.empty: raise ValueError("The embedding files contain no shared validation molecules")

fingerprint_cache = load_or_compute_fingerprints(
    FINGERPRINT_DIR, CHEMBL_DIR, SMILES_DIR, ids=library["ID"].to_numpy())
library["Molecule"] = [Chem.MolFromSmiles(smiles) for smiles in tqdm(fingerprint_cache["smiles"], desc="Preparing validation identities")]
invalid = library["Molecule"].isna()
if invalid.any(): raise ValueError(f"Cannot parse molecule ID {library.loc[invalid, 'ID'].iloc[0]}")
library["SMILES"] = [Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True) for mol in library["Molecule"]]
library_size = len(library)
library = library.drop_duplicates("SMILES", keep="first").reset_index(drop=True)
duplicates_removed = library_size - len(library)
if len(library) < 4: raise ValueError("At least four unique validation molecules are required")
query_rows = match_molecule_ids(library["ID"].to_numpy(), query_ids).numpy()

embeddings = {name: normalize_embeddings(cache["embeddings"][library[name].to_numpy()]) for name, cache in caches.items()}
del caches, cache, cache_ids
fingerprint_rows = match_molecule_ids(fingerprint_cache["id"], library["ID"].to_numpy())
fingerprints = {}
for label, fp_type in (("ECFP", "ecfp"), ("RDKFP", "rdkit")):
    packed = fingerprint_cache[fp_type][fingerprint_rows].numpy().view(np.uint8)
    # Cached fingerprints use MSB-first bytes; RDKit binary text uses LSB-first bytes.
    binary = np.packbits(np.unpackbits(packed, axis=1), axis=1, bitorder="little")
    fingerprints[label] = [DataStructs.CreateFromBinaryText(row.tobytes()) for row in binary]
del fingerprint_cache, fingerprint_rows, packed, binary
library = library[["ID", "SMILES"]]
print(f"{len(library):,} unique validation molecules; {len(query_rows):,} queries shared with Figure 3; "
      f"{duplicates_removed:,} duplicate identities excluded.")

query_records = []
all_rows = np.arange(len(library))
with tqdm(total=len(query_rows), desc="Held-out query comparisons") as progress:
    for start in range(0, len(query_rows), query_batch_size):
        queries = query_rows[start:start + query_batch_size]
        cosine = {name: np.clip(values[queries] @ values.T, -1, 1) for name, values in embeddings.items()}
        for local_row, query in enumerate(queries):
            candidates = all_rows != query
            fingerprint_ranks = {label: normalize_ranks(np.asarray(DataStructs.BulkTanimotoSimilarity(bitvectors[query], bitvectors))[candidates]) for label, bitvectors in fingerprints.items()}
            ecfp, rdkfp = fingerprint_ranks["ECFP"], fingerprint_ranks["RDKFP"]
            record = {"ID": int(library.iloc[query]["ID"]), "SMILES": library.iloc[query]["SMILES"],
                      "Candidates": int(candidates.sum()),
                      "ECFP vs RDKFP": rank_correlation(ecfp, rdkfp)}
            for name, similarities in cosine.items():
                ranks = normalize_ranks(similarities[local_row, candidates])
                record[f"{name} vs ECFP"] = rank_correlation(ranks, ecfp)
                record[f"{name} vs RDKFP"] = rank_correlation(ranks, rdkfp)
                record[f"{name} vs RDKFP | ECFP"] = partial_rank_correlation(ranks, rdkfp, ecfp)
            query_records.append(record)
            progress.update(1)
query_results = pd.DataFrame(query_records)

comparisons = ([f"{name} vs {label}" for name in embeddings for label in fingerprints]
               + ["ECFP vs RDKFP"]
               + [f"{name} vs RDKFP | ECFP" for name in embeddings])
correlation_records = [{"Comparison": comparison, **bootstrap_median_interval(query_results[comparison], n_bootstrap, seed)} for comparison in comparisons]
paired_records = []
for other in ("NT", "nopre"):
    for label in ("ECFP", "RDKFP", "RDKFP | ECFP"):
        differences = query_results[f"ANT vs {label}"] - query_results[f"{other} vs {label}"]
        paired_records.append({"Comparison": f"ANT - {other}: {label}", **bootstrap_median_interval(differences, n_bootstrap, seed)})

## Validation Library

In [ ]:
display(pd.DataFrame([
    {"Model": name, "Library molecules": len(library)}
for name in embeddings]))

## Table S5: Validation Fingerprint Correlations

In [ ]:
display(pd.DataFrame(correlation_records).rename(columns={
    "n": "Queries", "median": "Median", "lower": "95% CI lower", "upper": "95% CI upper",
}).round(4))

## Table S6: Paired Correlation Differences

In [ ]:
display(pd.DataFrame(paired_records).rename(columns={
    "n": "Queries", "median": "Median", "lower": "95% CI lower", "upper": "95% CI upper",
}).round(4))